In [2]:
import astropy.units as u
from astropy.constants import c, eps0, mu0
from numpy import sqrt, pi

# Skin Depth

Im trying to answer how much energy the wave loses by the skin effect in a metal conductor/reflector.
I assume the dish will be build from aluminium.

The length perpendicular to the surface of a conductor where the magintude of the electric field has dropped of by a fact of $\frac{1}{e}$. 

According to Wikipedia its given bz 
$$
\delta= \sqrt{{2\rho }\over{\omega\mu}} \; \; \sqrt{ \sqrt{1 + \left({\rho\omega\epsilon}\right)^2 } + \rho\omega\epsilon}
$$

where $\rho$ is the resistivity of the material and $\omega$ is the angular frequency of the incident wave and
$$
\epsilon = \epsilon_{r} \epsilon_{0}
$$
where $\epsilon_r$ is the relative permittivity of the material and $\epsilon_{0}$ is the permittivity of free space.
The magnetic permeability of aluminum is close to 1.

$$
\mu = \mu_{r} \mu_{0}
$$

Pretending we have an aluminum dish we get the following material constants (at room temperature)

$\rho = 2.65×10^{-8} \Omega m $ 

I don't really know what the value for $\epsilon_r$ is in aluminium. This is the most conservative number I found. 
$\epsilon_r = 1.8 $ 


In [11]:
eps_r = 1.8
eps = eps0 * eps_r
mu_r = 1.0
mu = mu0 * mu_r

In [12]:
rho = 2.65e-8 * u.ohm * u.m

In [13]:
f = 10 * u.GHz
w = 2 * pi * f
wavelength = c / f

Assuming an incident wave of 10 GHz we get for the second radical in the equation

$$
a = \sqrt{ \sqrt{1 + b^2 } + b}
$$

where $b = \left({\rho\omega\epsilon}\right)$

In [14]:
b = w * rho * eps
a = sqrt(sqrt(1 + b**2) + b)

In [15]:
skin_depth = sqrt(2 * rho / (w * mu)) * a
skin_depth.to("um")

<Quantity 0.81930023 um>

In [16]:
t = skin_depth / wavelength
t.si

<Quantity 2.73289138e-05>

This leads to a skin depth of a few micro meters. Which is only a millionth of the wavelength.
I conclude that almost no energy is lost in the reflection. 

# Estimating reflectivity

Sebastian here. I am not sure but I would assume that the fraction of the energy which does not get absorbed will be reflected. Maybe this can be used to estimate the reflectivity of the aluminium layer versus the thisckness of the aluminium layer.

In [65]:
factors = np.geomspace(0.1, 10, 11)

In [66]:
reflectivity = 1.0 - 1 / np.exp(factors)

In [67]:
reflector_thickness = skin_depth.to("m") * factors

In [68]:
from matplotlib import pyplot as plt

In [84]:
_scale = 4
fig = plt.Figure(figsize=(_scale*1.6, _scale*0.9), dpi=1000/_scale)
ax = fig.add_axes([0.15, 0.15, 0.8, 0.8])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(color="grey", linestyle="-", linewidth=0.33, which="major")
ax.grid(color="lightgrey", linestyle="-", linewidth=0.15, which="minor")
ax.plot(reflector_thickness * 1e6, reflectivity, color="black")
ax.loglog()
ax.axvline(skin_depth.to("um").to_value(), linestyle="--", color="gray")
ax.text(0.55, 0.5, "skin depth", transform=ax.transAxes, color="gray")
ax.set_ylabel("reflectivity at 10GHz / 1")
ax.set_xlabel("thickness of aluminium coating / $\mu$m")
fig.savefig("reflectivity_of_thin_aluminium_coating.jpg")